# 03 — Deep Blue data: voxel-wise kinetics with the uptake–washout model

Notebook 02 fitted the uptake–washout model to the total activity of each organ and lesion. Here the **same model** is fitted to the time-activity curve of every voxel, giving a TIA map and maps of the kinetic parameters on the scan-1 grid. This is the voxel-level version of the physics layer of the project: the network will predict these parameters from the pre-therapy PET, and the layer will turn them into $A(t)$ and TIA.

$$A(\mathbf{x}, t) = A_0(\mathbf{x})\left(e^{-\lambda_w(\mathbf{x}) t} - e^{-(\lambda_w(\mathbf{x}) + \lambda_u(\mathbf{x}))\, t}\right), \qquad \lambda_w = \lambda_\text{phys} + \lambda_\text{bio},\ \lambda_\text{bio} \ge 0,\ \lambda_u > 0$$

$$\text{TIA}(\mathbf{x}) = A_0\left(\frac{1}{\lambda_w} - \frac{1}{\lambda_w + \lambda_u}\right)$$

**Steps**
1. Align scans 2–4 to scan 1 voxel by voxel (structure-wise alignment) and check the conservation of the structure activities.
2. Fitting method (vectorised Levenberg–Marquardt).
3. Fit on two grids: 2×2×2 binning (3.9 mm, main result) and native voxels (1.95 mm, sensitivity analysis).
4. Quality of the fits.
5. Comparison with the organ-level fits of notebook 02 and with the MIM TIA map, on both grids.
6. Save the maps for the dose calculation (notebook 04).

**About this notebook.** Structure-wise alignment and voxel-wise uptake-washout fits on two grids.
- **Inputs:** nb01, nb02 outputs.
- **Outputs:** `data/processed/deepblue/<patient>/voxel/`, `kinetics/voxel_*.csv`.
- **Run time:** tens of minutes (not recorded).
- **Environment:** .venv.

In [ ]:
import json
from concurrent.futures import ThreadPoolExecutor
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import SimpleITK as sitk

WS = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "paths.py").exists())
sys.path.insert(0, str(WS / "src"))
from paths import DATASET
PROC = WS / "data" / "processed" / "deepblue"
KIN = PROC / "kinetics"
PATIENTS = ["patient_4", "patient_6"]
SNMMI_LABEL = {"patient_4": "A", "patient_6": "B"}

In [ ]:
def read(path):
    return sitk.ReadImage(str(path))


def arr(image):
    return sitk.GetArrayFromImage(image)


def voxel_ml(image):
    return float(np.prod(image.GetSpacing())) / 1000.0


def like(array, reference):
    image = sitk.GetImageFromArray(array)
    image.CopyInformation(reference)
    return image


def centroid(mask):
    idx = np.argwhere(arr(mask) > 0).mean(axis=0)[::-1]
    return np.array(mask.TransformContinuousIndexToPhysicalPoint(idx.tolist()))


def dice(a, b):
    a, b = arr(a) > 0, arr(b) > 0
    return 2.0 * (a & b).sum() / (a.sum() + b.sum())


def onto(image, reference, transform, nearest=False, default=0.0):
    interp = sitk.sitkNearestNeighbor if nearest else sitk.sitkLinear
    return sitk.Resample(image, reference, transform, interp, default, image.GetPixelID())


data = {}
for patient in PATIENTS:
    pdir = PROC / patient
    timing = json.loads((pdir / "timing.json").read_text())
    scans = {}
    for scan, t_h in sorted(timing["t_h"].items(), key=lambda kv: kv[1]):
        sdir = pdir / scan
        scans[scan] = {
            "t_h": t_h,
            "spect": read(sdir / "spect_bqml.nii.gz"),
            "ct": read(sdir / "ct_on_spect.nii.gz"),
            "masks": {p.name.removesuffix(".nii.gz").split("_", 1)[1]: read(p)
                      for p in sorted((sdir / "masks").glob("*.nii.gz"))},
        }
    data[patient] = {"timing": timing, "scans": scans, "tia_mim": read(pdir / "tia_mim_bqml_s.nii.gz")}

LAM_PHYS = np.log(2) / (data[PATIENTS[0]]["timing"]["half_life_s"] / 3600.0)
tac02 = pd.read_csv(KIN / "tac.csv")
fits02 = pd.read_csv(KIN / "fits_all_models.csv")
fits02 = fits02[fits02.model == "uptake"].set_index(["patient", "structure"])
print({p: list(data[p]["scans"]["scan1"]["masks"]) for p in PATIENTS})

## 1. Alignment of the scans to scan 1

A voxel-wise fit needs the four images of the same piece of tissue in the same voxel. The patient was repositioned between scans, and the organs also move relative to each other and to the skeleton (breathing, bowel filling, posture).

**Why not a single global registration.** A global rigid CT-to-CT registration and a rigid + B-spline deformable registration (4 mm control grid, mutual information) were tested on these data. Neither aligns the kidneys and the lesions: for patient 6 the Dice coefficient between the scan-1 masks and the masks of the other scans mapped to scan 1 was 0.34–0.75 for the kidneys and as low as 0.0 for lesion 2 with the global rigid transform, and at most 0.49 for lesion 2 with the deformable one. The lesions are contoured on the SPECT and are not visible on the non-contrast CT, so a CT-driven registration cannot align them.

**Structure-wise alignment.** Each structure gets its own transform, the same idea as the contour-based alignment used by MIM for the reference TIA map:

| Region | Transform |
|---|---|
| liver | local rigid CT-to-CT registration (mutual information inside the scan-1 mask dilated by 16 mm, initialised with the centroid translation) |
| spleen | local rigid **SPECT-to-SPECT** registration inside the dilated scan-1 mask (as MIM did), adopted only if it beats the centroid translation (see below) |
| kidneys | local rigid CT registration **or** centroid translation, whichever gives the higher Dice for that kidney and scan (both are computed and reported) |
| lesions | translation that superimposes the centroids of the scan-1 and scan-$k$ masks |
| rest of the body | global rigid CT-to-CT registration |

The CT-based local rigid registration uses the anatomy, independent of how the contours were drawn on each scan, which matters for the liver (inconsistent contours, notebook 02). For the kidneys it was found to diverge occasionally in preliminary tests (Dice 0.42 for the right kidney of patient 6 at scan 3), so it is tested scan by scan against the centroid translation. Lesions are contoured on the SPECT and invisible on the CT, so only the translation is meaningful.

**Spleen.** Neither the centroid translation nor a CT-based local rigid registration aligned the spleen of patient 4: with the translation the aligned images lost 12% and 9% of the spleen activity at scans 2 and 3, and the CT registration was worse still (19% and 12%, Dice 0.71/0.66/0.86), because it is driven by the surrounding anatomy (ribs, stomach, colon), which moves differently from the spleen. The spleen is therefore registered on the **SPECT** itself: rigid transform, Mattes mutual information over all voxels of the scan-1 spleen mask dilated by 16 mm, initialised with the centroid translation. The uptake pattern changes little in shape over time, so the SPECT carries the information that the CT lacks. The SPECT registration is adopted only if, averaged over scans 2–4, it gives both a higher Dice and a smaller deviation of the activity conservation from 1 than the translation; otherwise the translation is kept and the residual is an alignment limitation.

**Assembly.** For every scan $k$ the aligned image on the scan-1 grid starts from the globally aligned SPECT; then the voxels of each scan-1 structure, dilated by 2 voxels (4 mm) to include the spill-out of the SPECT, are replaced by the SPECT aligned with that structure's transform. The order is rest of body → liver → kidneys and spleen → lesions, so lesions inside the liver use the lesion transform.

**Limitations.** The transforms are discontinuous at the borders of the regions; organ deformation is not modelled; resampling is done on the activity concentration without a Jacobian correction (the total activity of a structure is conserved only if its volume does not change, which is checked below).

In [ ]:
def translation(fixed_mask, moving_mask):
    t = sitk.Euler3DTransform()
    t.SetCenter(centroid(fixed_mask).tolist())
    t.SetTranslation((centroid(moving_mask) - centroid(fixed_mask)).tolist())
    return t


def body_mask(ct):
    return sitk.Cast(sitk.BinaryMorphologicalOpening(ct > -300, [2, 2, 2]), sitk.sitkUInt8)


def rigid(fixed_ct, moving_ct, fixed_roi, init, sampling=0.25):
    reg = sitk.ImageRegistrationMethod()
    reg.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
    reg.SetMetricFixedMask(fixed_roi)
    reg.SetMetricSamplingStrategy(reg.RANDOM)
    reg.SetMetricSamplingPercentage(sampling, 0)
    reg.SetInterpolator(sitk.sitkLinear)
    reg.SetOptimizerAsRegularStepGradientDescent(learningRate=2.0, minStep=1e-3, numberOfIterations=300,
                                                 relaxationFactor=0.5)
    reg.SetOptimizerScalesFromPhysicalShift()
    reg.SetShrinkFactorsPerLevel([4, 2, 1])
    reg.SetSmoothingSigmasPerLevel([2, 1, 0])
    reg.SetInitialTransform(init, inPlace=False)
    return reg.Execute(sitk.Cast(fixed_ct, sitk.sitkFloat32), sitk.Cast(moving_ct, sitk.sitkFloat32))


def global_rigid(fixed_ct, moving_ct):
    init = sitk.CenteredTransformInitializer(body_mask(fixed_ct), body_mask(moving_ct), sitk.Euler3DTransform(),
                                             sitk.CenteredTransformInitializerFilter.MOMENTS)
    return rigid(fixed_ct, moving_ct, body_mask(fixed_ct), init, sampling=0.1)


def local_rigid(fixed_ct, moving_ct, fixed_mask, moving_mask):
    return rigid(fixed_ct, moving_ct, sitk.BinaryDilate(fixed_mask, [8, 8, 8]), translation(fixed_mask, moving_mask))


ORDER = ["liver", "ltkidney", "rtkidney", "spleen"]
KIDNEYS = {"ltkidney", "rtkidney"}


def region_order(names):
    return [n for n in ORDER if n in names] + sorted(n for n in names if n.startswith("lesion"))


def spect_rigid(fixed_spect, moving_spect, fixed_mask, moving_mask):
    reg = sitk.ImageRegistrationMethod()
    reg.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
    reg.SetMetricFixedMask(sitk.BinaryDilate(fixed_mask, [8, 8, 8]))
    reg.SetMetricSamplingStrategy(reg.NONE)
    reg.SetInterpolator(sitk.sitkLinear)
    reg.SetOptimizerAsRegularStepGradientDescent(learningRate=1.0, minStep=1e-3, numberOfIterations=300,
                                                 relaxationFactor=0.5)
    reg.SetOptimizerScalesFromPhysicalShift()
    reg.SetShrinkFactorsPerLevel([2, 1])
    reg.SetSmoothingSigmasPerLevel([1, 0])
    reg.SetInitialTransform(translation(fixed_mask, moving_mask), inPlace=False)
    return reg.Execute(sitk.Cast(fixed_spect, sitk.sitkFloat32), sitk.Cast(moving_spect, sitk.sitkFloat32))


def conservation_ratio(patient, n, scan, s, ref, transform):
    a = float(arr(onto(s["spect"], ref["spect"], transform))[arr(ref["masks"][n]) > 0].sum()) * voxel_ml(ref["spect"]) / 1e6
    ref_name = "liver_whole" if n == "liver" else n
    return a / tac02[(tac02.patient == patient) & (tac02.structure == ref_name) & (tac02.scan == scan)].activity_MBq.iloc[0]


align_qc, spleen_test = [], []
for patient in PATIENTS:
    scans = data[patient]["scans"]
    ref = scans["scan1"]
    names = region_order(ref["masks"])
    regions = {n: arr(sitk.BinaryDilate(ref["masks"][n], [2, 2, 2])) > 0 for n in names}
    labels = np.zeros(arr(ref["spect"]).shape, np.uint8)
    for i, n in enumerate(names, start=1):
        labels[regions[n]] = i
    data[patient]["region_labels"] = like(labels, ref["spect"])
    data[patient]["region_names"] = names
    transforms = {}
    for scan, s in list(scans.items())[1:]:
        mapped_dice = lambda n, tr: dice(ref["masks"][n], onto(s["masks"][n], ref["spect"], tr, nearest=True))
        transforms[scan] = {"body": global_rigid(ref["ct"], s["ct"])}
        for n in names:
            candidates = {"translation": translation(ref["masks"][n], s["masks"][n])}
            if n == "liver" or n in KIDNEYS:
                candidates["local rigid CT"] = local_rigid(ref["ct"], s["ct"], ref["masks"][n], s["masks"][n])
            if n == "spleen":
                candidates["local rigid SPECT"] = spect_rigid(ref["spect"], s["spect"], ref["masks"][n], s["masks"][n])
            d = {k: mapped_dice(n, tr) for k, tr in candidates.items()}
            if n == "spleen":
                for k, tr in candidates.items():
                    spleen_test.append({"patient": patient, "scan": scan, "transform": k, "dice": d[k],
                                        "conservation": conservation_ratio(patient, n, scan, s, ref, tr)})
            used = ("local rigid CT" if n == "liver" else max(d, key=d.get) if n in KIDNEYS else "translation")
            transforms[scan][n] = candidates
            align_qc.append({"patient": patient, "structure": n, "scan": scan,
                             "dice_global_rigid": mapped_dice(n, transforms[scan]["body"]),
                             "dice_translation": d["translation"], "dice_local_rigid_CT": d.get("local rigid CT", np.nan),
                             "dice_local_rigid_SPECT": d.get("local rigid SPECT", np.nan),
                             "transform_used": used})
    if "spleen" in names:
        test = pd.DataFrame([r for r in spleen_test if r["patient"] == patient])
        test["conservation_error"] = (test.conservation - 1).abs()
        mean = test.groupby("transform")[["dice", "conservation_error"]].mean()
        better = (mean.loc["local rigid SPECT", "dice"] > mean.loc["translation", "dice"]
                  and mean.loc["local rigid SPECT", "conservation_error"] < mean.loc["translation", "conservation_error"])
        spleen_choice = "local rigid SPECT" if better else "translation"
        for row in align_qc:
            if row["patient"] == patient and row["structure"] == "spleen":
                row["transform_used"] = spleen_choice
        print(patient, "spleen transform:", spleen_choice)
    for row in align_qc:
        if row["patient"] == patient:
            transforms[row["scan"]][row["structure"]] = transforms[row["scan"]][row["structure"]][row["transform_used"]]
    for scan, s in scans.items():
        if scan == "scan1":
            s["aligned"] = ref["spect"]
            s["covered"] = np.ones(labels.shape, bool)
            continue
        T = transforms[scan]
        ones = like(np.ones(arr(s["spect"]).shape, np.uint8), s["spect"])
        aligned = arr(onto(s["spect"], ref["spect"], T["body"]))
        covered = arr(onto(ones, ref["spect"], T["body"], nearest=True)) > 0
        for n in names:
            aligned[regions[n]] = arr(onto(s["spect"], ref["spect"], T[n]))[regions[n]]
            covered[regions[n]] = (arr(onto(ones, ref["spect"], T[n], nearest=True)) > 0)[regions[n]]
        s["aligned"] = like(aligned.astype(np.float32), ref["spect"])
        s["covered"] = covered
    print(patient, "aligned")

spleen_test = pd.DataFrame(spleen_test)
spleen_test.pivot_table(index=["patient", "transform"], columns="scan", values=["dice", "conservation"]).round(3)

In [ ]:
align_qc = pd.DataFrame(align_qc).set_index(["patient", "structure", "scan"])
align_qc.round(2)

**Conservation of the structure activities.** The activity inside each scan-1 mask of the aligned image of scan $k$ is compared with the activity inside the mask drawn on scan $k$ in its own image (the TACs of notebook 02; for the liver the scan-1 reference TAC of notebook 02, which used the same transform). Ratios close to 1 mean that the aligned voxel curves carry the right total activity; deviations reflect differences between the contours of the two scans and residual misalignment, and propagate directly to the voxel TIA of that structure.

In [ ]:
conservation = []
for patient in PATIENTS:
    scans = data[patient]["scans"]
    ref = scans["scan1"]
    for n in data[patient]["region_names"]:
        m = arr(ref["masks"][n]) > 0
        ref_name = "liver_whole" if n == "liver" else n
        for scan, s in scans.items():
            a_aligned = float(arr(s["aligned"])[m].sum()) * voxel_ml(ref["spect"]) / 1e6
            a03 = tac02[(tac02.patient == patient) & (tac02.structure == ref_name) & (tac02.scan == scan)].activity_MBq.iloc[0]
            conservation.append({"patient": patient, "structure": n, "scan": scan, "ratio_aligned_over_nb02": a_aligned / a03})
conservation = pd.DataFrame(conservation)
conservation.pivot_table(index=["patient", "structure"], columns="scan", values="ratio_aligned_over_nb02").round(3)

Visual check: coronal slice of the aligned SPECT at the four time points (shared colour scale per patient), with the scan-1 contours of the structures. The activity should stay inside the same contours while it decays.

In [ ]:
for patient in PATIENTS:
    scans = data[patient]["scans"]
    labels = arr(data[patient]["region_labels"])
    organs = np.isin(labels, [i + 1 for i, n in enumerate(data[patient]["region_names"]) if not n.startswith("lesion")])
    y = int(np.argwhere(organs).mean(axis=0)[1])
    vmax = np.percentile(arr(scans["scan1"]["aligned"])[:, y, :], 99.9)
    fig, axes = plt.subplots(1, len(scans), figsize=(4 * len(scans), 4.5))
    for ax, (scan, s) in zip(axes, scans.items()):
        ax.imshow(arr(s["aligned"])[::-1, y, :], cmap="inferno", vmax=vmax, aspect="auto")
        ax.contour(labels[::-1, y, :] > 0, [0.5], colors="cyan", linewidths=0.6)
        ax.set_title(f"{scan}, t = {s['t_h']:.1f} h")
        ax.axis("off")
    fig.suptitle(f"{patient}: aligned SPECT on the scan-1 grid, coronal slice")
    fig.tight_layout()

## 2. Voxel-wise fit: method

**Voxels fitted.** Every voxel of the body (scan-1 CT above −300 HU, after a morphological opening) that is covered by all four aligned scans. Voxels outside the field of view of at least one scan cannot be fitted and get no TIA; their fraction is reported.

**Fit.** The model, bounds and weighting are those of notebook 02, so that organ-level and voxel-level results are directly comparable:
- parameters $A_0$, $\lambda_\text{bio} \ge 0$ and $\lambda_u > 0$, fitted in logarithmic form ($A_0 = e^{u_0}$, $\lambda_\text{bio} = e^{u_1}$, $\lambda_u = e^{u_2}$), which enforces the bounds without a constrained optimiser; $u_0$ is limited to $[-30, 50]$ to avoid overflows in empty voxels; $\lambda_\text{bio}$ is limited to $[e^{-20}, 2]\ \text{h}^{-1}$ and $\lambda_u$ to $[10^{-4}, 20]\ \text{h}^{-1}$;
- relative residuals, with a floor: $\sigma_i = \max(A_i,\ 0.05 \max_j A_j)$. The floor prevents near-zero or negative voxel values (noise in low-activity regions) from receiving an infinite weight;
- Levenberg–Marquardt written in NumPy and vectorised over voxels, with the 3×3 normal equations solved in closed form; three starting values of the uptake half-life (1, 6 and 24 h), keeping the best fit per voxel; voxels stop iterating when the relative decrease of the cost is below $10^{-6}$.

The implementation reproduces the `lmfit` organ-level fits of notebook 02 to better than 0.1% in TIA for all structures but one (`liver_whole_perscan` of patient 4, 3%, where the two fits ended in two different, almost equivalent minima).

**Units.** Activities in Bq/mL and time in hours give TIA in Bq·h/mL; the maps are converted to Bq·s/mL, the unit of the MIM map.

In [ ]:
U0_MIN, U0_MAX = -30.0, 50.0
U1_MIN, U1_MAX, U2_MIN, U2_MAX = -20.0, np.log(2.0), np.log(1e-4), np.log(20.0)


def _eval(u, t, y, sigma):
    a0, lb, lu = np.exp(u[:, 0]), np.exp(u[:, 1]), np.exp(u[:, 2])
    lw = LAM_PHYS + lb
    e1, e2 = np.exp(-lw[:, None] * t), np.exp(-(lw + lu)[:, None] * t)
    model = a0[:, None] * (e1 - e2)
    J = np.empty(y.shape + (3,))
    J[..., 0] = model / sigma
    J[..., 1] = a0[:, None] * t * (e2 - e1) * lb[:, None] / sigma
    J[..., 2] = a0[:, None] * t * e2 * lu[:, None] / sigma
    return (model - y) / sigma, J


def _clip(u):
    u[:, 0] = np.clip(u[:, 0], U0_MIN, U0_MAX)
    u[:, 1] = np.clip(u[:, 1], U1_MIN, U1_MAX)
    u[:, 2] = np.clip(u[:, 2], U2_MIN, U2_MAX)
    return u


def _init(t, y, sigma, lam_up):
    yp = np.maximum(y, 1e-12)
    lb = np.clip(np.log(yp[:, -2] / yp[:, -1]) / (t[-1] - t[-2]) - LAM_PHYS, np.exp(U1_MIN), 2.0)
    lw = LAM_PHYS + lb
    f = np.exp(-lw[:, None] * t) - np.exp(-(lw + lam_up)[:, None] * t)
    w = 1 / sigma**2
    a0 = np.maximum((f * y * w).sum(1) / (f * f * w).sum(1), 1e-12)
    return np.column_stack([np.log(a0), np.log(lb), np.full(len(y), np.log(lam_up))])


def _solve3(a, g):
    a00, a01, a02, a11, a12, a22 = a
    c00, c01, c02 = a11 * a22 - a12 * a12, a02 * a12 - a01 * a22, a01 * a12 - a02 * a11
    c11, c12, c22 = a00 * a22 - a02 * a02, a01 * a02 - a00 * a12, a00 * a11 - a01 * a01
    det = a00 * c00 + a01 * c01 + a02 * c02
    det = np.where(np.abs(det) > 1e-300, det, 1e-300)
    return np.stack([(c00 * g[0] + c01 * g[1] + c02 * g[2]) / det,
                     (c01 * g[0] + c11 * g[1] + c12 * g[2]) / det,
                     (c02 * g[0] + c12 * g[1] + c22 * g[2]) / det], axis=1)


def _fit_from(t, y, sigma, lam_up0, n_iter, tol):
    u = _clip(_init(t, y, sigma, lam_up0))
    mu = np.full(len(y), 1e-2)
    r, J = _eval(u, t, y, sigma)
    cost = (r**2).sum(1)
    active = np.arange(len(y))
    for _ in range(n_iter):
        if active.size == 0:
            break
        ua, ra, Ja, ca, ma = u[active], r[active], J[active], cost[active], mu[active]
        j0, j1, j2 = Ja[..., 0], Ja[..., 1], Ja[..., 2]
        d = [(j0 * j0).sum(1), (j0 * j1).sum(1), (j0 * j2).sum(1), (j1 * j1).sum(1), (j1 * j2).sum(1), (j2 * j2).sum(1)]
        g = [-(j0 * ra).sum(1), -(j1 * ra).sum(1), -(j2 * ra).sum(1)]
        a = [d[0] * (1 + ma) + 1e-12, d[1], d[2], d[3] * (1 + ma) + 1e-12, d[4], d[5] * (1 + ma) + 1e-12]
        un = _clip(ua + _solve3(a, g))
        rn, Jn = _eval(un, t, y[active], sigma[active])
        cn = (rn**2).sum(1)
        ok = np.isfinite(cn) & (cn < ca)
        idx = active[ok]
        u[idx], r[idx], J[idx], cost[idx] = un[ok], rn[ok], Jn[ok], cn[ok]
        mu[active] = np.clip(np.where(ok, ma * 0.3, ma * 10.0), 1e-9, 1e9)
        converged = (ok & ((ca - cn) <= tol * (ca + 1e-12))) | (~ok & (mu[active] >= 1e9))
        active = active[~converged]
    return u, cost


def fit_uptake(t, y, rel_floor=0.05, n_iter=60, tol=1e-6, lam_up_starts=np.log(2) / np.array([1.0, 6.0, 24.0])):
    sigma = np.maximum(np.maximum(y, rel_floor * y.max(axis=1, keepdims=True)), 1e-12)
    best_u, best_cost = None, None
    for lu0 in lam_up_starts:
        u, cost = _fit_from(t, y, sigma, lu0, n_iter, tol)
        if best_u is None:
            best_u, best_cost = u, cost
        else:
            better = cost < best_cost
            best_u[better], best_cost[better] = u[better], cost[better]
    a0, lb, lu = np.exp(best_u.T)
    return {"a0": a0, "lam_bio": lb, "lam_up": lu, "rms_rel_residual": np.sqrt(best_cost / y.shape[1])}


def fit_voxels(t, y, chunk=100_000, workers=8):
    parts = [y[i:i + chunk] for i in range(0, len(y), chunk)]
    with ThreadPoolExecutor(workers) as ex:
        res = list(ex.map(lambda part: fit_uptake(t, part), parts))
    return {k: np.concatenate([r[k] for r in res]) for k in res[0]}


def tia_uptake(a0, lam_bio, lam_up, t_from=0.0):
    lw = LAM_PHYS + lam_bio
    lu = lw + lam_up
    return a0 * (np.exp(-lw * t_from) / lw - np.exp(-lu * t_from) / lu)

## 3. Two grids: native voxels and 2×2×2 binning

The voxel curves are noisy, especially for patient 4, whose images are reconstructed without a post-filter (`xSPECT48iNOFILT`) and show a blobby noise texture that changes from scan to scan. The model is therefore fitted on two grids:

| Grid | Voxel | Role |
|---|---|---|
| `bin2` | 3.9 mm (2×2×2 native voxels averaged) | **main result**; matches the 4 mm voxels of the GATE dose kernel and is still finer than the spatial resolution of Lu-177 SPECT |
| `native` | 1.95 mm | sensitivity analysis |

Binning averages the activity concentration of the 8 native voxels of each bin, so the activity in every bin is conserved; it is applied to the aligned SPECT of each scan before the fit. With 199 slices, the last native slice is dropped by the binning.

**Correlated noise.** For uncorrelated noise, averaging 8 voxels would reduce the noise by a factor $\sqrt{8} \approx 2.8$. Here it does not: the residuals of the fit are practically the same on the two grids (see the quality table). The noise of these reconstructions is spatially correlated over 1–2 cm, as the blobby texture in the liver of patient 4 shows, so neighbouring native voxels carry almost the same noise realisation and averaging them removes little of it. Reducing it would need a scale comparable to the correlation length (a filter of that width, or super-voxels inside organs), which would also remove real spatial detail. The voxel noise is **not tuned further** in this work: the `bin2` grid is kept because it matches the dose kernel, and the residual noise is part of the uncertainty of the voxel-level TIA.

On the binned grid, structures are represented by their **fraction maps**: the fraction of each bin that lies inside the scan-1 mask. Structure sums are $\sum_\text{bins} \text{TIA} \times f \times V_\text{bin}$, and voxel-level statistics use the bins with $f \ge 0.5$. The MIM map is binned in the same way for the comparison. A bin is fitted if at least half of it lies in the native fit mask.

In [ ]:
BIN = [2, 2, 2]


def binned(image):
    return sitk.BinShrink(sitk.Cast(image, sitk.sitkFloat32), BIN)


def fit_grid(aligned, fit_mask, t):
    y = np.stack([a[fit_mask] for a in aligned], axis=1).astype(float)
    res = fit_voxels(t, y)
    tia = tia_uptake(res["a0"], res["lam_bio"], res["lam_up"])
    values = {"tia_bqml_s": tia * 3600.0, "a0_bqml": res["a0"],
              "t_eff_h": np.log(2) / (LAM_PHYS + res["lam_bio"]),
              "t_uptake_h": np.log(2) / (LAM_PHYS + res["lam_bio"] + res["lam_up"]),
              "lam_bio_at_bound": res["lam_bio"] < 1e-6, "rms_rel_residual": res["rms_rel_residual"],
              "tail_fraction": tia_uptake(res["a0"], res["lam_bio"], res["lam_up"], t[-1]) / np.maximum(tia, 1e-30)}
    maps = {}
    for k, v in values.items():
        maps[k] = np.zeros(fit_mask.shape, np.float32)
        maps[k][fit_mask] = v
    qc = {"voxels_fitted": int(fit_mask.sum()),
          "lam_bio_at_bound_%": 100 * (res["lam_bio"] < 1e-6).mean(),
          "lam_up_at_upper_bound_%": 100 * (res["lam_up"] > 19.9).mean(),
          "median_rms_rel_residual": float(np.median(res["rms_rel_residual"]))}
    return maps, qc


GRIDS = ["bin2", "native"]
fit_qc = []
for patient in PATIENTS:
    scans = data[patient]["scans"]
    ref = scans["scan1"]
    t = np.array([s["t_h"] for s in scans.values()])
    body = arr(body_mask(ref["ct"])) > 0
    covered = np.all([s["covered"] for s in scans.values()], axis=0)
    fit_native = body & covered
    ref_b = binned(ref["spect"])
    grids = {
        "native": {"reference": ref["spect"], "fit_mask": fit_native,
                   "aligned": [arr(s["aligned"]) for s in scans.values()],
                   "fractions": {n: (arr(ref["masks"][n]) > 0).astype(np.float32) for n in data[patient]["region_names"]},
                   "tia_mim": arr(data[patient]["tia_mim"]),
                   "labels": arr(data[patient]["region_labels"])},
        "bin2": {"reference": ref_b, "fit_mask": arr(binned(like(fit_native.astype(np.float32), ref["spect"]))) >= 0.5,
                 "aligned": [arr(binned(s["aligned"])) for s in scans.values()],
                 "fractions": {n: arr(binned(ref["masks"][n])) for n in data[patient]["region_names"]},
                 "tia_mim": arr(binned(data[patient]["tia_mim"])),
                 "labels": arr(sitk.Resample(data[patient]["region_labels"], ref_b, sitk.Transform(),
                                             sitk.sitkNearestNeighbor, 0))},
    }
    for g in GRIDS:
        grids[g]["maps"], qc = fit_grid(grids[g]["aligned"], grids[g]["fit_mask"], t)
        fit_qc.append({"patient": patient, "grid": g, "voxel_mm": grids[g]["reference"].GetSpacing()[0], **qc})
    data[patient]["grids"] = grids
    fit_qc.append({"patient": patient, "grid": "native", "body_voxels_not_covered_%": 100 * (body & ~covered).sum() / body.sum()})
fit_qc = pd.DataFrame(fit_qc).groupby(["patient", "grid"]).first()
fit_qc.round(3)

## 4. Quality of the voxel fits

Per structure and grid: median and 90th percentile of the RMS relative residual, median effective and uptake half-lives, fraction of voxels with $\lambda_\text{bio}$ on its lower bound ($T_\text{eff} = T_\text{phys}$) and median fraction of the TIA extrapolated after the last scan. `body` is every fitted voxel; structure voxels are those with fraction ≥ 0.5.

Voxels with $\lambda_\text{bio}$ on the bound are expected in two situations: slow true washout (as for lesion 2 of patient 4 at the organ level) and noisy or misaligned voxels whose last two points do not decrease. A high fraction inside an organ points to the second case, and should drop with binning.

In [ ]:
quality = []
for patient in PATIENTS:
    for g in GRIDS:
        G = data[patient]["grids"][g]
        regions = {"body": G["fit_mask"]}
        regions.update({n: (f >= 0.5) & G["fit_mask"] for n, f in G["fractions"].items()})
        for n, m in regions.items():
            mp = {k: v[m] for k, v in G["maps"].items()}
            quality.append({"patient": patient, "structure": n, "grid": g, "voxels": int(m.sum()),
                            "rms_res_median": np.median(mp["rms_rel_residual"]),
                            "rms_res_p90": np.percentile(mp["rms_rel_residual"], 90),
                            "T_eff_median_h": np.median(mp["t_eff_h"]),
                            "T_uptake_median_h": np.median(mp["t_uptake_h"]),
                            "lam_bio_at_bound_%": 100 * mp["lam_bio_at_bound"].mean(),
                            "tail_fraction_median": np.median(mp["tail_fraction"])})
quality = pd.DataFrame(quality).set_index(["patient", "structure", "grid"])
quality.unstack("grid")[["rms_res_median", "rms_res_p90", "lam_bio_at_bound_%", "T_eff_median_h"]].round(3)

Parameter maps on the `bin2` grid (main result), coronal slice through the organs: TIA, effective half-life, uptake half-life, RMS relative residual. The last panel shows the native-grid residual for comparison. Voxels outside the fit mask are blank.

In [ ]:
panels = [("bin2", "tia_bqml_s", "TIA [Bq s/mL], bin2", "inferno", None),
          ("bin2", "t_eff_h", "T_eff [h], bin2", "viridis", (0, 160)),
          ("bin2", "t_uptake_h", "T_uptake [h], bin2", "viridis", (0, 24)),
          ("bin2", "rms_rel_residual", "RMS rel. residual, bin2", "magma", (0, 0.3)),
          ("native", "rms_rel_residual", "RMS rel. residual, native", "magma", (0, 0.3))]


def coronal_index(G, names):
    organs = np.isin(G["labels"], [i + 1 for i, n in enumerate(names) if not n.startswith("lesion")])
    return int(np.argwhere(organs).mean(axis=0)[1])


for patient in PATIENTS:
    names = data[patient]["region_names"]
    fig, axes = plt.subplots(1, len(panels), figsize=(4.0 * len(panels), 4.5))
    for ax, (g, key, title, cmap, lim) in zip(axes, panels):
        G = data[patient]["grids"][g]
        y = coronal_index(G, names)
        img = np.where(G["fit_mask"], G["maps"][key], np.nan)[::-1, y, :]
        vmin, vmax = lim if lim else (0, np.nanpercentile(img, 99.9))
        im = ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
        ax.contour(G["labels"][::-1, y, :] > 0, [0.5], colors="white", linewidths=0.5)
        ax.set_title(title, fontsize=10)
        ax.axis("off")
        fig.colorbar(im, ax=ax, fraction=0.046)
    fig.suptitle(f"{patient} (SNMMI patient {SNMMI_LABEL[patient]})")
    fig.tight_layout()

## 5. Comparison with the organ-level fits and with MIM

For every structure and grid, three TIAs are compared:
- **voxel**: sum of the voxel TIA map over the structure (fraction-weighted on the binned grid);
- **organ (nb02)**: TIA of the uptake model fitted to the organ TAC in notebook 02;
- **MIM**: sum of the MIM TIA map over the same structure on the same grid.

The sum of voxel-wise TIAs is not the TIA of the summed curve: the model is non-linear, so the two differ even with perfect alignment, especially where the voxel curves are noisy; binning should bring the voxel sum closer to the organ-level value. In addition the organ-level TAC of notebook 02 uses the mask drawn on each scan, while here every time point uses the scan-1 mask.

In [ ]:
comparison = []
for patient in PATIENTS:
    for g in GRIDS:
        G = data[patient]["grids"][g]
        v = voxel_ml(G["reference"])
        tv, tm = G["maps"]["tia_bqml_s"], G["tia_mim"]
        for n, f in G["fractions"].items():
            comparison.append({"patient": patient, "structure": n, "grid": g,
                               "TIA_voxel_MBq_h": (tv * f).sum() * v / 3.6e9,
                               "TIA_nb02_MBq_h": fits02.loc[(patient, "liver_whole" if n == "liver" else n), "TIA_MBq_h"],
                               "TIA_MIM_MBq_h": (tm * f).sum() * v / 3.6e9})
        fm = G["fit_mask"]
        comparison.append({"patient": patient, "structure": "fitted body", "grid": g,
                           "TIA_voxel_MBq_h": tv[fm].sum() * v / 3.6e9, "TIA_nb02_MBq_h": np.nan,
                           "TIA_MIM_MBq_h": tm[fm].sum() * v / 3.6e9})
comparison = pd.DataFrame(comparison).set_index(["patient", "structure", "grid"])
comparison["voxel_over_nb02"] = comparison.TIA_voxel_MBq_h / comparison.TIA_nb02_MBq_h
comparison["voxel_over_MIM"] = comparison.TIA_voxel_MBq_h / comparison.TIA_MIM_MBq_h
comparison.to_csv(KIN / "voxel_tia_comparison.csv")
comparison[["voxel_over_nb02", "voxel_over_MIM"]].unstack("grid").round(3)

**Voxel-by-voxel agreement with MIM.** Inside each structure (fraction ≥ 0.5), on each grid: median of the voxel ratio ours/MIM, its interquartile range and the Pearson correlation of the two maps. The two maps were built with different registrations, so a voxel-level spread is expected at the borders and in small lesions even when the totals agree; binning should reduce the part of the spread due to noise and to one-voxel misalignments.

In [ ]:
voxel_agreement = []
for patient in PATIENTS:
    for g in GRIDS:
        G = data[patient]["grids"][g]
        tv, tm = G["maps"]["tia_bqml_s"], G["tia_mim"]
        for n, f in G["fractions"].items():
            ok = (f >= 0.5) & G["fit_mask"] & (tm > 0)
            ratio = tv[ok] / tm[ok]
            voxel_agreement.append({"patient": patient, "structure": n, "grid": g, "voxels": int(ok.sum()),
                                    "median_ratio": np.median(ratio), "ratio_p25": np.percentile(ratio, 25),
                                    "ratio_p75": np.percentile(ratio, 75), "pearson_r": np.corrcoef(tv[ok], tm[ok])[0, 1]})
voxel_agreement = pd.DataFrame(voxel_agreement).set_index(["patient", "structure", "grid"])
voxel_agreement.to_csv(KIN / "voxel_agreement_mim.csv")
voxel_agreement[["median_ratio", "ratio_p25", "ratio_p75", "pearson_r"]].unstack("grid").round(3)

In [ ]:
for patient in PATIENTS:
    names = data[patient]["region_names"]
    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    for row, g in zip(axes, GRIDS):
        G = data[patient]["grids"][g]
        y = coronal_index(G, names)
        fm, tv, tm, labels = G["fit_mask"], G["maps"]["tia_bqml_s"], G["tia_mim"], G["labels"]
        vmax = np.percentile(tm[fm], 99.9)
        row[0].imshow(np.where(fm, tv, np.nan)[::-1, y, :], cmap="inferno", vmin=0, vmax=vmax, aspect="auto")
        row[0].set_title(f"voxel fit, {g}")
        row[1].imshow(np.where(fm, tm, np.nan)[::-1, y, :], cmap="inferno", vmin=0, vmax=vmax, aspect="auto")
        row[1].set_title(f"MIM, {g}")
        ratio = np.full(tv.shape, np.nan)
        hot = fm & (tm > 0.05 * vmax)
        ratio[hot] = tv[hot] / tm[hot]
        im = row[2].imshow(ratio[::-1, y, :], cmap="RdBu_r", vmin=0.5, vmax=1.5, aspect="auto")
        row[2].set_title("ratio voxel / MIM (MIM > 5% of max)")
        fig.colorbar(im, ax=row[2], fraction=0.046)
        for ax in row[:3]:
            ax.contour(labels[::-1, y, :] > 0, [0.5], colors="cyan", linewidths=0.5)
            ax.axis("off")
        sel = fm & (tm > 0.01 * vmax) & (tv > 0)
        row[3].hexbin(tm[sel], tv[sel], gridsize=100, bins="log", xscale="log", yscale="log", mincnt=1, cmap="viridis")
        lim = [np.percentile(tm[sel], 1), tm[sel].max()]
        row[3].plot(lim, lim, "r-", lw=0.8)
        row[3].set_xlabel("MIM TIA [Bq s/mL]")
        row[3].set_ylabel("voxel-fit TIA [Bq s/mL]")
        row[3].set_title(f"{g}: voxels with MIM > 1% of max")
    fig.suptitle(f"{patient} (SNMMI patient {SNMMI_LABEL[patient]})")
    fig.tight_layout()

### Findings

- **Spleen of patient 4: the SPECT-driven registration solves the alignment.** Against the centroid translation it raises the Dice at scans 2 and 3 (0.87 and 0.79 against 0.77 and 0.74; 0.88 at scan 4 for both) and brings the activity conservation close to 1 (0.96, 0.99, 1.05 against 0.88, 0.91, 1.03), so it is adopted by the rule. The voxel TIA sum of the spleen rises from 0.87 to 0.94 of the organ-level fit and from 0.86 to 0.94 of MIM; inside the spleen the median RMS residual falls from 0.20 to 0.12, the fraction of voxels with $\lambda_\text{bio}$ on its bound from 15% to 1.3%, and the voxel correlation with MIM rises from 0.79 to 0.98. A CT-based local rigid registration had been tried first and was worse than the translation (Dice 0.71/0.66/0.86, 19% and 12% of the activity lost at scans 2 and 3): the CT is dominated by the anatomy around the spleen, the SPECT by the spleen itself. The remaining 4–5% deficit and the 5% excess at scan 4 are left as an alignment limitation (a rigid transform cannot follow the deformation of the organ).
- **Kidneys: local rigid CT and translation are equivalent.** The local rigid registration is chosen in 7 of the 12 kidney–scan cases, always with a Dice gain of at most 0.02, and loses by up to 0.03 in the other 5. The kidney sums are within 2% of the translation-only version.
- **Correlated noise.** Binning 2×2×2 does not reduce the voxel noise (liver of patient 4: median RMS residual 0.30 on `bin2`, 0.31 native), because the noise of these reconstructions is correlated over 1–2 cm (section 3). The noise is not tuned further and is part of the uncertainty of the voxel TIA.
- **Structure sums on the two grids** agree within 1–2% for organs and large lesions. For the small lesions the `bin2` sum is lower (lesion 2: 0.86 against 0.90 of the organ-level fit for patient 4, 0.94 against 0.98 for patient 6): bins on the lesion border mix the lesion curve with the liver curve, and the TIA of a mixed curve fitted with a non-linear model is not the mixture of the TIAs.
- **Comparison with MIM (`bin2`)**: kidneys 0.94–1.04, liver 0.91 (patient 4) and 1.05 (patient 6), spleen 0.94, lesions 0.99–1.10 except lesion 2 (1.25 for patient 4, 1.19 for patient 6, as at the organ level); fitted body 0.91 (patient 4) and 1.00 (patient 6). The native grid gives the same values within 1–4%.
- **Voxels with near-zero fitted TIA.** About 18 000 native voxels of patient 4 and 1 800 of patient 6 get a TIA close to zero where MIM is above 1% of its maximum (bottom of the scatter plots). Almost all lie outside the structures and have at least one zero sample: outside the reconstructed field of one scan, or activity that moves through the bowel (e.g. 0, 42, 153, 0 kBq/mL at the four scans). Such curves are not described by an uptake–washout model; they hold 0.76% (patient 4) and 0.12% (patient 6) of the MIM TIA of the fitted body.
- **Voxel level.** Agreement of the totals does not imply agreement voxel by voxel: for the left kidney of patient 6 the totals agree within 1% but the voxel correlation with MIM is only r = 0.17. A shift of one or two voxels in a small organ with a steep activity gradient is enough to decorrelate two maps built with different alignments.
- **Liver of patient 4 and lesion 2 of patient 4** behave as before: noisy voxel curves in the liver ($\lambda_\text{bio}$ on its bound in 10% of the bins), and $\lambda_\text{bio}$ on the bound in about 60% of the voxels of lesion 2 with about 60% of its TIA extrapolated after the last scan.

## 6. Saving

In `data/processed/deepblue/<patient>/voxel/`:
- `spect_aligned_<scan>.nii.gz`: physical activity concentration of each scan aligned to scan 1 (Bq/mL, native grid);
- `regions.nii.gz` and `regions.json`: label map of the alignment regions (native grid);
- for each grid `<g>` in `bin2` (3.9 mm, main result) and `native` (1.95 mm, sensitivity analysis):
  - `fit_mask_<g>.nii.gz`: voxels fitted;
  - `tia_bqml_s_<g>.nii.gz`: voxel TIA (Bq·s/mL, same unit as the MIM map); `tia_bqml_s_bin2.nii.gz` is the source for the dose calculation of notebook 04;
  - `a0_bqml_<g>`, `t_eff_h_<g>`, `t_uptake_h_<g>`, `lam_bio_at_bound_<g>`, `rms_rel_residual_<g>`, `tail_fraction_<g>` (`.nii.gz`): parameter and quality maps.

And in `data/processed/deepblue/kinetics/`: `voxel_tia_comparison.csv` and `voxel_agreement_mim.csv`.

In [ ]:
for patient in PATIENTS:
    vdir = PROC / patient / "voxel"
    vdir.mkdir(parents=True, exist_ok=True)
    for scan, s in data[patient]["scans"].items():
        sitk.WriteImage(s["aligned"], str(vdir / f"spect_aligned_{scan}.nii.gz"))
    sitk.WriteImage(data[patient]["region_labels"], str(vdir / "regions.nii.gz"))
    (vdir / "regions.json").write_text(json.dumps({i + 1: n for i, n in enumerate(data[patient]["region_names"])}, indent=2))
    for g in GRIDS:
        G = data[patient]["grids"][g]
        sitk.WriteImage(like(G["fit_mask"].astype(np.uint8), G["reference"]), str(vdir / f"fit_mask_{g}.nii.gz"))
        for k, v in G["maps"].items():
            sitk.WriteImage(like(v, G["reference"]), str(vdir / f"{k}_{g}.nii.gz"))
    print("saved", vdir)